# esmoe quick start

[esmoe](https://github.com/Lfan-ke/ES-MoE) adds an ES-MoE (expert-sparse Mixture-of-Experts) block to
Ultralytics YOLO. It installs *beside* the official `ultralytics` package - no fork, no patched
library - and it wires the router's load-balancing loss into the optimised training loss.

This notebook runs end to end in a few minutes on a free Colab GPU:

1. install and check versions
2. equip a model in one call
3. train and watch the `esmoe_aux` column move
4. same-budget on/off comparison, with an honest caveat
5. swap in your own expert and balancing objective
6. multi-point grafting and the command line

In [ ]:
try:
    import esmoe
except ImportError:
    !pip install -q esmoe
    import esmoe

import torch
import ultralytics

print(f"esmoe {esmoe.__version__} | ultralytics {ultralytics.__version__} | torch {torch.__version__}")
print("cuda:", torch.cuda.is_available())

## 1. Equip a model in one call

`equip` registers the block so a config can name it, grafts it onto the backbone (renumbering every
head reference that the insertion would otherwise break), builds the model and attaches the
auxiliary loss.

In [ ]:
import esmoe

model = esmoe.equip("yolo11n.yaml", weight=0.01)
block = next(esmoe.blocks(model.model))
print(block)

## 2. Train, and watch the auxiliary loss

`coco8` is Ultralytics' 8-image toy dataset - it downloads in seconds and is meant for plumbing
checks, not for accuracy. What matters here is the `esmoe_aux` column: it is non-zero, it is part of
the loss that gets back-propagated, and it moves.

In [ ]:
import pandas as pd

model.train(
    data="coco8.yaml",
    epochs=3,
    imgsz=320,
    batch=4,
    workers=2,
    plots=False,
    name="quickstart-esmoe",
    exist_ok=True,
    verbose=False,
)

frame = pd.read_csv(model.trainer.save_dir / "results.csv")
print([c for c in frame.columns if "esmoe" in c])
frame[[c for c in frame.columns if "loss" in c or "esmoe" in c]]

## 3. Same-budget on/off

Identical data, schedule, batch and seed; only the block differs. On eight images the gap is noise -
the point is that the comparison is mechanically fair and reproducible, not that the number means
anything. Real evidence lives in
[docs/SELECTION.md](https://github.com/Lfan-ke/ES-MoE/blob/main/docs/SELECTION.md): three seeds on
VisDrone under one budget.

In [ ]:
from ultralytics import YOLO

baseline = YOLO("yolo11n.yaml")
baseline.train(
    data="coco8.yaml",
    epochs=3,
    imgsz=320,
    batch=4,
    workers=2,
    plots=False,
    name="quickstart-baseline",
    exist_ok=True,
    verbose=False,
)

for name, trained in (("baseline", baseline), ("esmoe", model)):
    print(f"{name:9s} mAP50={trained.trainer.metrics['metrics/mAP50(B)']:.4f}")
print("\nEight images. Treat this as a plumbing check, not as evidence.")

## 4. Bring your own expert and balancing objective

Both are plain callables, so a variant is a few lines. `expert(c1, c2, k) -> Module`,
`balance(probs, gate) -> scalar`.

In [ ]:
import torch
from torch import nn


class ThinExpert(nn.Sequential):
    def __init__(self, c1, c2, k):
        super().__init__(nn.Conv2d(c1, c2, k, 1, k // 2, groups=c1), nn.SiLU())


def entropy_balance(probs, gate):
    return -(probs * probs.clamp_min(1e-9).log()).sum(dim=1).mean()


custom = esmoe.ESMoE(num_experts=3, top_k=2, expert=ThinExpert, balance=entropy_balance)
custom(torch.randn(2, 32, 16, 16))
print("aux:", esmoe.collect_aux_loss(custom).item())
print("experts:", [type(e).__name__ for e in custom.experts])

## 5. Several blocks, and the command line

`at` takes the backbone end (default), one index, or several. References are renumbered for every
insertion.

In [ ]:
cfg = esmoe.graft("yolo11n.yaml", at=[4, 6], num_experts=4, top_k=2)
print([i for i, layer in enumerate(cfg["backbone"]) if layer[2] == "ESMoE"])

!esmoe graft yolo11n.yaml -o yolo11n-esmoe.yaml -e 4 -k 2 --at backbone_end
!esmoe info

## Where to go next

- [Documentation](https://lfan-ke.github.io/ES-MoE/)
- [Selection evidence](https://github.com/Lfan-ke/ES-MoE/blob/main/docs/SELECTION.md) - why 4 experts,
  top-2 and an aux weight of 0.01 are the shipped defaults
- [Limitations](https://github.com/Lfan-ke/ES-MoE/blob/main/limitations.md) - read before quoting any
  number from this project
- `scripts/sweep.sh` in the repository reproduces the multi-seed comparison on a real dataset